[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/algoritmos-UdeA/logica1-20262/blob/main/clases/15/teoria/notebooks/sala_archivo_paso_a_paso.ipynb) [![Built with AI](https://img.shields.io/badge/Built%20with-AI-blue.svg)](https://shields.io/)

# La sala de cine que se guarda, paso a paso

Este notebook construye el bloque 2 de la [clase 15](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/15/README.md): la sala de cine de la clase 14 aprende a **guardarse** en un archivo y a **cargarse** de él. Así, al cerrar el programa, las ventas no se pierden.

Se parte de la `Sala` de la clase 14 y se le agrega, en dos pasos, el método `guardar` y después el método `cargar` y la función `cargar_sala`. El código es el de [`sala_archivo/`](https://github.com/algoritmos-UdeA/logica1-20262/tree/main/clases/15/teoria/codigo/sala_archivo).

> 📁 Los archivos que crea el notebook quedan en la carpeta de trabajo (en Colab, en el almacenamiento temporal de la sesión).

## Cómo usar este notebook

1. Se puede abrir en Google Colab con el botón **Open in Colab** de arriba, sin instalar nada, o en Jupyter (Anaconda Navigator o la extensión de Jupyter de VS Code).
2. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con **Shift + Enter**: casi todas usan lo definido en celdas anteriores.
3. Las salidas ya están guardadas, pero conviene **predecir** qué va a imprimir cada celda antes de ejecutarla.
4. Algunas celdas **fallan a propósito** (están marcadas con ⚠️). Por eso no conviene usar *ejecutar todo*: la ejecución se detiene en el primer error.
5. Si algo se comporta de forma extraña, se reinicia y se vuelve a ejecutar desde el principio: en Colab, **Entorno de ejecución → Reiniciar sesión**; en Jupyter, **Kernel → Restart Kernel**.

| Marca | Significado |
|---|---|
| 🔮 **Prediga** | Antes de ejecutar la celda, responda en papel qué va a pasar |
| ⚠️ **Error frecuente** | La celda siguiente falla o da un resultado equivocado **a propósito** |
| ✏️ **Hágalo usted mismo** | Un ejercicio; la solución está oculta y se despliega con un clic |
| 🔎 **Python Tutor** | Debajo de las celdas que no usan archivos: abre ese código en [Python Tutor](https://pythontutor.com/). **Python Tutor no permite abrir archivos**: por eso las celdas que leen o escriben archivos no tienen enlace, y este notebook muestra el contenido de cada archivo con `mostrar_archivo` |

---

## Parte 0 — Preparar

Dos ayudas para el notebook: `crear_archivo` y `mostrar_archivo`.

In [ ]:
def crear_archivo(nombre_archivo, contenido):
    """Garantiza: CREA (o REEMPLAZA) el archivo con el texto dado."""
    archivo = open(nombre_archivo, "w")
    archivo.write(contenido)
    archivo.close()


def mostrar_archivo(nombre_archivo):
    """Garantiza: muestra el contenido del archivo, tal como esta en el disco."""
    archivo = open(nombre_archivo, "r")
    contenido = archivo.read()
    print(f"----- {nombre_archivo} -----")
    print(contenido, end="")
    if not contenido.endswith("\n"):
        print()
    print("-" * (len(nombre_archivo) + 12))
    archivo.close()

---

## Parte 1 — El problema: la sala que olvida

Las clases `Silla` y `Sala` de la clase 14, sin los comentarios de documentación (están completos en el notebook [Sala de cine paso a paso](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/14/teoria/notebooks/sala_cine_paso_a_paso.ipynb)):

In [ ]:
class Silla:

    def __init__(self, fila, numero):
        self.__fila = fila
        self.__numero = numero
        self.__ocupada = False

    def get_fila(self):
        return self.__fila

    def get_numero(self):
        return self.__numero

    def esta_libre(self):
        return not self.__ocupada

    def ocupar(self):
        ocupada_ahora = False
        if not self.__ocupada:
            self.__ocupada = True
            ocupada_ahora = True
        return ocupada_ahora

    def liberar(self):
        liberada = False
        if self.__ocupada:
            self.__ocupada = False
            liberada = True
        return liberada

    def __str__(self):
        texto = "O"
        if self.__ocupada:
            texto = "X"
        return texto

In [ ]:
class Sala:

    LETRAS = "ABCDEFGHIJKL"

    def __init__(self, filas, sillas_por_fila, precio):
        self.__filas = filas
        self.__sillas_por_fila = sillas_por_fila
        self.__precio = precio
        self.__vendidas = 0
        self.__sillas = [None] * filas
        for i in range(filas):
            self.__sillas[i] = [None] * sillas_por_fila
            for j in range(sillas_por_fila):
                self.__sillas[i][j] = Silla(Sala.LETRAS[i], j + 1)

    def get_filas(self):
        return self.__filas

    def get_sillas_por_fila(self):
        return self.__sillas_por_fila

    def indice_fila(self, letra):
        indice = -1
        for i in range(self.__filas):
            if Sala.LETRAS[i] == letra:
                indice = i
        return indice

    def es_silla_valida(self, letra, numero):
        return self.indice_fila(letra) != -1 and 1 <= numero <= self.__sillas_por_fila

    def get_silla(self, letra, numero):
        return self.__sillas[self.indice_fila(letra)][numero - 1]

    def vender(self, letra, numero):
        vendida = False
        if self.es_silla_valida(letra, numero):
            if self.get_silla(letra, numero).ocupar():
                self.__vendidas = self.__vendidas + 1
                vendida = True
        return vendida

    def liberar(self, letra, numero):
        liberada = False
        if self.es_silla_valida(letra, numero):
            if self.get_silla(letra, numero).liberar():
                self.__vendidas = self.__vendidas - 1
                liberada = True
        return liberada

    def contar_libres_fila(self, letra):
        i = self.indice_fila(letra)
        libres = 0
        for j in range(self.__sillas_por_fila):
            if self.__sillas[i][j].esta_libre():
                libres = libres + 1
        return libres

    def contar_libres(self):
        libres = 0
        for i in range(self.__filas):
            libres = libres + self.contar_libres_fila(Sala.LETRAS[i])
        return libres

    def get_recaudo(self):
        return self.__vendidas * self.__precio

    def __str__(self):
        texto = "  "
        for j in range(self.__sillas_por_fila):
            texto = texto + f" {j + 1}"
        for i in range(self.__filas):
            texto = texto + "\n" + Sala.LETRAS[i] + " "
            for j in range(self.__sillas_por_fila):
                texto = texto + " " + str(self.__sillas[i][j])
        return texto

In [ ]:
sala = Sala(5, 6, 5000)
sala.vender("A", 1)
sala.vender("C", 4)
print(sala)
print("Recaudo: $", sala.get_recaudo())

   1 2 3 4 5 6
A  X O O O O O
B  O O O O O O
C  O O O X O O
D  O O O O O O
E  O O O O O O
Recaudo: $ 10000


🔎 [Ver paso a paso en Python Tutor](https://pythontutor.com/visualize.html#code=class%20Silla%3A%0A%0A%20%20%20%20def%20__init__%28self%2C%20fila%2C%20numero%29%3A%0A%20%20%20%20%20%20%20%20self.__fila%20%3D%20fila%0A%20%20%20%20%20%20%20%20self.__numero%20%3D%20numero%0A%20%20%20%20%20%20%20%20self.__ocupada%20%3D%20False%0A%0A%20%20%20%20def%20get_fila%28self%29%3A%0A%20%20%20%20%20%20%20%20return%20self.__fila%0A%0A%20%20%20%20def%20get_numero%28self%29%3A%0A%20%20%20%20%20%20%20%20return%20self.__numero%0A%0A%20%20%20%20def%20esta_libre%28self%29%3A%0A%20%20%20%20%20%20%20%20return%20not%20self.__ocupada%0A%0A%20%20%20%20def%20ocupar%28self%29%3A%0A%20%20%20%20%20%20%20%20ocupada_ahora%20%3D%20False%0A%20%20%20%20%20%20%20%20if%20not%20self.__ocupada%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20self.__ocupada%20%3D%20True%0A%20%20%20%20%20%20%20%20%20%20%20%20ocupada_ahora%20%3D%20True%0A%20%20%20%20%20%20%20%20return%20ocupada_ahora%0A%0A%20%20%20%20def%20liberar%28self%29%3A%0A%20%20%20%20%20%20%20%20liberada%20%3D%20False%0A%20%20%20%20%20%20%20%20if%20self.__ocupada%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20self.__ocupada%20%3D%20False%0A%20%20%20%20%20%20%20%20%20%20%20%20liberada%20%3D%20True%0A%20%20%20%20%20%20%20%20return%20liberada%0A%0A%20%20%20%20def%20__str__%28self%29%3A%0A%20%20%20%20%20%20%20%20texto%20%3D%20%22O%22%0A%20%20%20%20%20%20%20%20if%20self.__ocupada%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20texto%20%3D%20%22X%22%0A%20%20%20%20%20%20%20%20return%20texto%0A%0Aclass%20Sala%3A%0A%0A%20%20%20%20LETRAS%20%3D%20%22ABCDEFGHIJKL%22%0A%0A%20%20%20%20def%20__init__%28self%2C%20filas%2C%20sillas_por_fila%2C%20precio%29%3A%0A%20%20%20%20%20%20%20%20self.__filas%20%3D%20filas%0A%20%20%20%20%20%20%20%20self.__sillas_por_fila%20%3D%20sillas_por_fila%0A%20%20%20%20%20%20%20%20self.__precio%20%3D%20precio%0A%20%20%20%20%20%20%20%20self.__vendidas%20%3D%200%0A%20%20%20%20%20%20%20%20self.__sillas%20%3D%20%5BNone%5D%20%2A%20filas%0A%20%20%20%20%20%20%20%20for%20i%20in%20range%28filas%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20self.__sillas%5Bi%5D%20%3D%20%5BNone%5D%20%2A%20sillas_por_fila%0A%20%20%20%20%20%20%20%20%20%20%20%20for%20j%20in%20range%28sillas_por_fila%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20self.__sillas%5Bi%5D%5Bj%5D%20%3D%20Silla%28Sala.LETRAS%5Bi%5D%2C%20j%20%2B%201%29%0A%0A%20%20%20%20def%20indice_fila%28self%2C%20letra%29%3A%0A%20%20%20%20%20%20%20%20indice%20%3D%20-1%0A%20%20%20%20%20%20%20%20for%20i%20in%20range%28self.__filas%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20if%20Sala.LETRAS%5Bi%5D%20%3D%3D%20letra%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20indice%20%3D%20i%0A%20%20%20%20%20%20%20%20return%20indice%0A%0A%20%20%20%20def%20es_silla_valida%28self%2C%20letra%2C%20numero%29%3A%0A%20%20%20%20%20%20%20%20return%20self.indice_fila%28letra%29%20%21%3D%20-1%20and%201%20%3C%3D%20numero%20%3C%3D%20self.__sillas_por_fila%0A%0A%20%20%20%20def%20get_silla%28self%2C%20letra%2C%20numero%29%3A%0A%20%20%20%20%20%20%20%20return%20self.__sillas%5Bself.indice_fila%28letra%29%5D%5Bnumero%20-%201%5D%0A%0A%20%20%20%20def%20vender%28self%2C%20letra%2C%20numero%29%3A%0A%20%20%20%20%20%20%20%20vendida%20%3D%20False%0A%20%20%20%20%20%20%20%20if%20self.es_silla_valida%28letra%2C%20numero%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20if%20self.get_silla%28letra%2C%20numero%29.ocupar%28%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20self.__vendidas%20%3D%20self.__vendidas%20%2B%201%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20vendida%20%3D%20True%0A%20%20%20%20%20%20%20%20return%20vendida%0A%0A%20%20%20%20def%20get_recaudo%28self%29%3A%0A%20%20%20%20%20%20%20%20return%20self.__vendidas%20%2A%20self.__precio%0A%0A%20%20%20%20def%20__str__%28self%29%3A%0A%20%20%20%20%20%20%20%20texto%20%3D%20%22%20%20%22%0A%20%20%20%20%20%20%20%20for%20j%20in%20range%28self.__sillas_por_fila%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20texto%20%3D%20texto%20%2B%20f%22%20%7Bj%20%2B%201%7D%22%0A%20%20%20%20%20%20%20%20for%20i%20in%20range%28self.__filas%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20texto%20%3D%20texto%20%2B%20%22%5Cn%22%20%2B%20Sala.LETRAS%5Bi%5D%20%2B%20%22%20%22%0A%20%20%20%20%20%20%20%20%20%20%20%20for%20j%20in%20range%28self.__sillas_por_fila%29%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20texto%20%3D%20texto%20%2B%20%22%20%22%20%2B%20str%28self.__sillas%5Bi%5D%5Bj%5D%29%0A%20%20%20%20%20%20%20%20return%20texto%0A%0Asala%20%3D%20Sala%285%2C%206%2C%205000%29%0Asala.vender%28%22A%22%2C%201%29%0Asala.vender%28%22C%22%2C%204%29%0Aprint%28sala%29%0Aprint%28%22Recaudo%3A%20%24%22%2C%20sala.get_recaudo%28%29%29&cumulative=false&curInstr=0&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=311&rawInputLstJSON=%5B%5D&textReferences=false)

> ⚠️ Cuando el programa termina, sus objetos desaparecen. Aquí se **simula** cerrar el programa borrando la variable con `del`:

In [ ]:
del sala
print(sala)

NameError: name 'sala' is not defined

Al volver a empezar, habría que crear otra sala, que nace vacía: las ventas de A1 y C4 se perdieron.

![Dos ejecuciones: lo guardado en la primera sigue en la segunda](https://raw.githubusercontent.com/algoritmos-UdeA/logica1-20262/main/clases/15/teoria/images/sala_ejecuciones.png)

---

## Parte 2 — Decidir el formato del archivo

Antes de programar se decide **qué** se guarda y **cómo**. Es el contrato entre `guardar` y `cargar`:

* **Primera línea:** el tamaño y el precio, separados por comas: `5,6,5000`.
* **Una línea por fila** de la sala, con una letra por silla: `X` ocupada, `O` libre. Es justamente lo que retorna `str(silla)`.

![La sala en memoria y sala.txt en el disco](https://raw.githubusercontent.com/algoritmos-UdeA/logica1-20262/main/clases/15/teoria/images/sala_archivo_formato.png)

---

## Parte 3 — Paso 1: `guardar`

Es un **recorrido por filas** de la matriz de objetos. Cada silla se escribe con `str(silla)` y, al terminar cada fila, un `"\n"`. La clase completa, ahora con `guardar` (el único método con su documentación):

In [ ]:
class Sala:

    LETRAS = "ABCDEFGHIJKL"

    def __init__(self, filas, sillas_por_fila, precio):
        self.__filas = filas
        self.__sillas_por_fila = sillas_por_fila
        self.__precio = precio
        self.__vendidas = 0
        self.__sillas = [None] * filas
        for i in range(filas):
            self.__sillas[i] = [None] * sillas_por_fila
            for j in range(sillas_por_fila):
                self.__sillas[i][j] = Silla(Sala.LETRAS[i], j + 1)

    def get_filas(self):
        return self.__filas

    def get_sillas_por_fila(self):
        return self.__sillas_por_fila

    def indice_fila(self, letra):
        indice = -1
        for i in range(self.__filas):
            if Sala.LETRAS[i] == letra:
                indice = i
        return indice

    def es_silla_valida(self, letra, numero):
        return self.indice_fila(letra) != -1 and 1 <= numero <= self.__sillas_por_fila

    def get_silla(self, letra, numero):
        return self.__sillas[self.indice_fila(letra)][numero - 1]

    def vender(self, letra, numero):
        vendida = False
        if self.es_silla_valida(letra, numero):
            if self.get_silla(letra, numero).ocupar():
                self.__vendidas = self.__vendidas + 1
                vendida = True
        return vendida

    def liberar(self, letra, numero):
        liberada = False
        if self.es_silla_valida(letra, numero):
            if self.get_silla(letra, numero).liberar():
                self.__vendidas = self.__vendidas - 1
                liberada = True
        return liberada

    def contar_libres_fila(self, letra):
        i = self.indice_fila(letra)
        libres = 0
        for j in range(self.__sillas_por_fila):
            if self.__sillas[i][j].esta_libre():
                libres = libres + 1
        return libres

    def contar_libres(self):
        libres = 0
        for i in range(self.__filas):
            libres = libres + self.contar_libres_fila(Sala.LETRAS[i])
        return libres

    def get_recaudo(self):
        return self.__vendidas * self.__precio

    def guardar(self, nombre_archivo):
        """
        Garantiza:
            CREA el archivo (o lo REEMPLAZA) con el tamano y el precio en
            la primera linea y una linea por fila de la sala, con el texto
            de cada silla (X u O). No modifica la sala. No retorna nada.
        """
        archivo = open(nombre_archivo, "w")
        archivo.write(f"{self.__filas},{self.__sillas_por_fila},{self.__precio}\n")
        for i in range(self.__filas):
            for j in range(self.__sillas_por_fila):
                archivo.write(str(self.__sillas[i][j]))
            archivo.write("\n")
        archivo.close()

    def __str__(self):
        texto = "  "
        for j in range(self.__sillas_por_fila):
            texto = texto + f" {j + 1}"
        for i in range(self.__filas):
            texto = texto + "\n" + Sala.LETRAS[i] + " "
            for j in range(self.__sillas_por_fila):
                texto = texto + " " + str(self.__sillas[i][j])
        return texto

In [ ]:
sala = Sala(5, 6, 5000)
sala.vender("A", 1)
sala.vender("B", 5)
sala.vender("B", 6)
sala.vender("C", 4)
sala.guardar("sala.txt")
mostrar_archivo("sala.txt")

----- sala.txt -----
5,6,5000
XOOOOO
OOOOXX
OOOXOO
OOOOOO
OOOOOO
--------------------


Se prueba leyendo el archivo línea por línea, con una sala pequeña:

In [ ]:
prueba = Sala(3, 4, 5000)
prueba.vender("A", 1)
prueba.vender("C", 4)
prueba.guardar("prueba_sala.txt")

archivo = open("prueba_sala.txt", "r")
assert archivo.readline() == "3,4,5000\n"
assert archivo.readline() == "XOOO\n"
assert archivo.readline() == "OOOO\n"
assert archivo.readline() == "OOOX\n"
assert archivo.readline() == ""
archivo.close()
assert prueba.contar_libres() == 10          # guardar no modifica la sala
print("Paso 1: todas las pruebas pasaron")

Paso 1: todas las pruebas pasaron


---

## Parte 4 — Paso 2: `cargar` y `cargar_sala`

`cargar` hace el camino contrario: se salta la primera línea (la sala ya conoce su tamaño), lee una línea por fila y, en cada silla, `"X"` → `ocupar()` y si no → `liberar()`. Vuelve a contar las ventas. **MODIFICA** la sala, y su *Supone* exige que el archivo tenga el mismo tamaño.

In [ ]:
class Sala:

    LETRAS = "ABCDEFGHIJKL"

    def __init__(self, filas, sillas_por_fila, precio):
        self.__filas = filas
        self.__sillas_por_fila = sillas_por_fila
        self.__precio = precio
        self.__vendidas = 0
        self.__sillas = [None] * filas
        for i in range(filas):
            self.__sillas[i] = [None] * sillas_por_fila
            for j in range(sillas_por_fila):
                self.__sillas[i][j] = Silla(Sala.LETRAS[i], j + 1)

    def get_filas(self):
        return self.__filas

    def get_sillas_por_fila(self):
        return self.__sillas_por_fila

    def indice_fila(self, letra):
        indice = -1
        for i in range(self.__filas):
            if Sala.LETRAS[i] == letra:
                indice = i
        return indice

    def es_silla_valida(self, letra, numero):
        return self.indice_fila(letra) != -1 and 1 <= numero <= self.__sillas_por_fila

    def get_silla(self, letra, numero):
        return self.__sillas[self.indice_fila(letra)][numero - 1]

    def vender(self, letra, numero):
        vendida = False
        if self.es_silla_valida(letra, numero):
            if self.get_silla(letra, numero).ocupar():
                self.__vendidas = self.__vendidas + 1
                vendida = True
        return vendida

    def liberar(self, letra, numero):
        liberada = False
        if self.es_silla_valida(letra, numero):
            if self.get_silla(letra, numero).liberar():
                self.__vendidas = self.__vendidas - 1
                liberada = True
        return liberada

    def contar_libres_fila(self, letra):
        i = self.indice_fila(letra)
        libres = 0
        for j in range(self.__sillas_por_fila):
            if self.__sillas[i][j].esta_libre():
                libres = libres + 1
        return libres

    def contar_libres(self):
        libres = 0
        for i in range(self.__filas):
            libres = libres + self.contar_libres_fila(Sala.LETRAS[i])
        return libres

    def get_recaudo(self):
        return self.__vendidas * self.__precio

    def guardar(self, nombre_archivo):
        """
        Garantiza:
            CREA el archivo (o lo REEMPLAZA) con el tamano y el precio en
            la primera linea y una linea por fila de la sala, con el texto
            de cada silla (X u O). No modifica la sala. No retorna nada.
        """
        archivo = open(nombre_archivo, "w")
        archivo.write(f"{self.__filas},{self.__sillas_por_fila},{self.__precio}\n")
        for i in range(self.__filas):
            for j in range(self.__sillas_por_fila):
                archivo.write(str(self.__sillas[i][j]))
            archivo.write("\n")
        archivo.close()

    def cargar(self, nombre_archivo):
        """
        Supone:
            el archivo existe, tiene el formato del modulo y el mismo tamano
            de esta sala.
        Garantiza:
            MODIFICA la sala: cada silla queda ocupada si en el archivo hay
            una X y libre si hay una O, y las ventas quedan contadas. No
            retorna nada.
        """
        archivo = open(nombre_archivo, "r")
        archivo.readline()                    # el tamano: ya lo conoce la sala
        self.__vendidas = 0
        for i in range(self.__filas):
            linea = archivo.readline().strip()
            for j in range(self.__sillas_por_fila):
                if linea[j] == "X":
                    self.__sillas[i][j].ocupar()
                    self.__vendidas = self.__vendidas + 1
                else:
                    self.__sillas[i][j].liberar()
        archivo.close()

    def __str__(self):
        texto = "  "
        for j in range(self.__sillas_por_fila):
            texto = texto + f" {j + 1}"
        for i in range(self.__filas):
            texto = texto + "\n" + Sala.LETRAS[i] + " "
            for j in range(self.__sillas_por_fila):
                texto = texto + " " + str(self.__sillas[i][j])
        return texto

`cargar_sala` es una **función** del módulo, no un método: lee solo la primera línea para saber el tamaño, crea la `Sala` y le pide que se cargue.

In [ ]:
def cargar_sala(nombre_archivo):
    """
    Supone:
        el archivo existe y tiene el formato del modulo.
    Garantiza:
        Retorna una Sala NUEVA con el tamano, el precio y las sillas
        ocupadas que dice el archivo.
    """
    archivo = open(nombre_archivo, "r")
    partes = archivo.readline().strip().split(",")
    archivo.close()
    sala = Sala(int(partes[0]), int(partes[1]), int(partes[2]))
    sala.cargar(nombre_archivo)
    return sala

---

## Parte 5 — Dos ejecuciones del programa

Se simulan las dos ejecuciones del dibujo de la Parte 1: en la primera se carga, se vende y se guarda; en la segunda se vuelve a cargar.

In [ ]:
# ===== Primera ejecucion =====
sala = cargar_sala("sala.txt")
print(sala)
sala.vender("D", 3)
sala.guardar("sala.txt")
print("Se vendio D3 y se guardo")
del sala                       # el programa termina

   1 2 3 4 5 6
A  X O O O O O
B  O O O O X X
C  O O O X O O
D  O O O O O O
E  O O O O O O
Se vendio D3 y se guardo


In [ ]:
# ===== Segunda ejecucion =====
sala = cargar_sala("sala.txt")
print(sala)
print("Recaudo: $", sala.get_recaudo())

   1 2 3 4 5 6
A  X O O O O O
B  O O O O X X
C  O O O X O O
D  O O X O O O
E  O O O O O O
Recaudo: $ 25000


D3 sigue ocupada y el recaudo cuenta las cinco boletas: la venta **sobrevivió** al programa. Es lo que hace `main.py`: `cargar_sala` al empezar y `guardar` al salir con la opción 0.

---

## Parte 6 — Pruebas de ida y vuelta

Si `guardar` y `cargar` son correctos, **guardar y después cargar** devuelve la misma sala:

In [ ]:
sala = Sala(3, 4, 5000)
sala.vender("A", 1)
sala.vender("C", 4)
sala.guardar("prueba_sala.txt")

otra = cargar_sala("prueba_sala.txt")
assert str(otra) == str(sala)
assert otra.get_recaudo() == 10000
assert not otra.get_silla("C", 4).esta_libre()

tercera = Sala(3, 4, 5000)
tercera.vender("B", 2)          # una venta que no esta en el archivo
tercera.cargar("prueba_sala.txt")
assert tercera.get_silla("B", 2).esta_libre()
assert str(tercera) == str(sala)
print("Pruebas de ida y vuelta: todas pasaron")

Pruebas de ida y vuelta: todas pasaron


---

## Parte 7 — Errores típicos

> ⚠️ **Error frecuente.** Cargar un archivo de **otro tamaño**. Si el archivo tiene menos filas que la sala, `readline` retorna `""` y `linea[j]` no existe:

In [ ]:
grande = Sala(5, 6, 5000)
grande.cargar("prueba_sala.txt")       # el archivo es de una sala de 3 x 4

IndexError: string index out of range

Y si el archivo es **más grande** que la sala, `cargar` no da error: lee solo lo que le cabe, y el resto del archivo se ignora sin avisar. Por eso existe `cargar_sala`, que crea la sala con el tamaño **del archivo**.

> ⚠️ **Error frecuente.** Guardar sobre el archivo equivocado. `guardar` abre con `"w"`: si el nombre es el de otro archivo, lo **reemplaza** sin preguntar.

In [ ]:
crear_archivo("importante.txt", "datos que no se deben perder\n")
Sala(2, 2, 5000).guardar("importante.txt")
mostrar_archivo("importante.txt")

----- importante.txt -----
2,2,5000
OO
OO
--------------------------


---

## Parte 8 — Para practicar

### ✏️ Ejercicio 1 — Ventas desde un archivo

Un archivo `ventas.txt` tiene una venta por línea, como `C4` o `A1`. Escriba una función `vender_desde_archivo(sala, nombre_archivo)` que venda cada silla del archivo y retorne cuántas ventas se pudieron hacer.

<details>
<summary>Ver solución</summary>

La letra es el primer carácter y el número es el resto: `linea[0]` y `int(linea[1:])`.

```python
def vender_desde_archivo(sala, nombre_archivo):
    vendidas = 0
    archivo = open(nombre_archivo, "r")
    for linea in archivo:
        linea = linea.strip()
        if sala.vender(linea[0], int(linea[1:])):
            vendidas = vendidas + 1
    archivo.close()
    return vendidas

crear_archivo("ventas.txt", "A1\nC4\nA1\nZ9\n")
sala = Sala(3, 4, 5000)
assert vender_desde_archivo(sala, "ventas.txt") == 2     # A1 repetida y Z9 no existe
```

</details>

### ✏️ Ejercicio 2 — Un resumen para el administrador

Escriba `guardar_resumen(sala, nombre_archivo)` que guarde, en un archivo de texto, una línea por fila con sus sillas libres (`Fila A: 3 libres`) y al final el recaudo.

<details>
<summary>Ver solución</summary>

```python
def guardar_resumen(sala, nombre_archivo):
    archivo = open(nombre_archivo, "w")
    for i in range(sala.get_filas()):
        letra = Sala.LETRAS[i]
        archivo.write(f"Fila {letra}: {sala.contar_libres_fila(letra)} libres\n")
    archivo.write(f"Recaudo: {sala.get_recaudo()}\n")
    archivo.close()

sala = Sala(2, 3, 5000)
sala.vender("A", 2)
guardar_resumen(sala, "resumen.txt")
archivo = open("resumen.txt", "r")
assert archivo.read() == "Fila A: 2 libres\nFila B: 3 libres\nRecaudo: 5000\n"
archivo.close()
```

</details>

---

## Parte 9 — Referencias

* [Clase 15](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/15/README.md) y su notebook [Archivos](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/15/teoria/notebooks/archivos.ipynb).
* [Clase 14](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/14/README.md) y el notebook [Sala de cine: una matriz de objetos](https://github.com/algoritmos-UdeA/logica1-20262/blob/main/clases/14/teoria/notebooks/sala_cine_paso_a_paso.ipynb).
* Harvard CS50P: [Lecture 6, File I/O](https://cs50.harvard.edu/python/notes/6/).